In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [3]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install konlpy
!pip install soynlp
!pip install sentenecepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.9/776.9 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.2/805.2 kB 36.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 81.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 42.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 6.6 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement sentenecepiece (from versions: none)
ERROR: No matching distribution found for sentenecepiece


In [4]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, LlamaForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=5
    batch_size=32
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'
bert_name = 'beomi/llama-2-ko-7b'

df_train = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_train.csv', encoding = 'utf-8')
df_dev = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_dev.csv', encoding = 'utf-8')
df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')

df_train.dropna(inplace = True)
df_dev.dropna(inplace = True)
df_test.dropna(inplace = True)

df_train.reset_index(drop = True, inplace = True)
df_dev.reset_index(drop = True, inplace = True)

cuda


In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, LlamaForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=5
    batch_size=32
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df= pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/테스트 데이터-스마일게이트/test_data.tsv', encoding = 'utf-8', sep = '\t')

cuda


In [ ]:
df.reset_index(drop = True, inplace = True)
df

,문장,clean,hate
0,불법체류 자체가 범죄인데 왜 자꾸 옹호하는 법안을 만들려고하냐저씨발년 누가 목 안따냐,0,1
1,게이가되야겠다,1,0
2,개좆같은 PC충들이랑 SJW 시발새끼들 부터 다 죽여버려야함,0,1
3,총기를 들고일어난 하나의 폭동이야,1,0
4,자집애들은 세상살기 참 편하노 후팔,0,1
...,...,...,...
2088,배는있냐 기름으로 움직이는거,1,0
2089,뿌리는 사람이나 벌리는 사람이나 지정신아님,1,0
2090,누군 자살을 종용했을까,1,0
2091,HQ 중국으로 옮기면 되겠다 ㅋ 중국 게임회사 좋네 ㅋ,1,0


In [ ]:
df['input'] = df['문장']
df['output'] = df['hate']

df = df.drop(['문장', 'clean', 'hate'], axis=1)

df

,input,output
0,불법체류 자체가 범죄인데 왜 자꾸 옹호하는 법안을 만들려고하냐저씨발년 누가 목 안따냐,1
1,게이가되야겠다,0
2,개좆같은 PC충들이랑 SJW 시발새끼들 부터 다 죽여버려야함,1
3,총기를 들고일어난 하나의 폭동이야,0
4,자집애들은 세상살기 참 편하노 후팔,1
...,...,...
2088,배는있냐 기름으로 움직이는거,0
2089,뿌리는 사람이나 벌리는 사람이나 지정신아님,0
2090,누군 자살을 종용했을까,0
2091,HQ 중국으로 옮기면 되겠다 ㅋ 중국 게임회사 좋네 ㅋ,0


In [ ]:
file_path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/신조어.txt'

In [ ]:
new_words_list = []

# 파일 열기 및 내용 읽기
with open(file_path, 'r', encoding='utf-8-sig') as file:
    for line in file:
        new_words_list.extend(line.strip().split(', '))

new_words_list

In [ ]:
from konlpy.tag import Okt
okt = Okt()

In [ ]:
# 이것, 저것, 그것, 이자, 저자, 그자, 이때, 그때, 저때 붙여쓰도록 하기
# 조사, 어미, 선어말어미, 접사의 경우 앞 말에 붙여쓰도록 하기
# 신조어의 경우, 신조어의 앞에 조사, 어미, 선어말어미, 접사가 붙어있다면 붙여쓰고, 신조어 뒤에 조사, 어미, 선어말어미가 붙어있다면 띄어쓰지 않도록 하기

def apply_morph_analysis(text, new_words_list):
    words = re.split('(\W+)', text)  # 비문자 구분자를 기준으로 분할

    corrected_text = ""
    for word in words:
        # 신조어가 아닌 경우에만 띄어쓰기 수정
        if word not in new_words_list:
            analyzed = okt.pos(word)
            for w, tag in analyzed:
                if tag in ['Josa', 'Eomi', 'PreEomi', 'Suffix']:
                    corrected_text += w
                else:
                    corrected_text += ' ' + w
        else:
            corrected_text += word

    return corrected_text.strip()

df['input'] = df['input'].apply(lambda x: apply_morph_analysis(x, new_words_list))

# df_train['input'] = df_train['input'].apply(lambda x: apply_morph_analysis(x, new_words_list))
# df_dev['input'] = df_dev['input'].apply(lambda x: apply_morph_analysis(x, new_words_list))
# df_test['input'] = df_test['input'].apply(lambda x: apply_morph_analysis(x, new_words_list))

In [ ]:
df

,input,output
0,불법체류 자체가 범죄인데 왜 자꾸 옹호 하는 법안을 만들려고하냐 저 씨발 년 누가 ...,1
1,게이가 되야겠다,0
2,개좆 같은 PC 충들이랑 SJW 시발 새끼들 부터 다 죽여 버려 야함,1
3,총기를 들고 일어난 하나의 폭동이야,0
4,자집애들은 세상 살기 참 편하노 후팔,1
...,...,...
2088,배는있냐 기름으로 움직이는거,0
2089,뿌리는 사람이나 벌리는 사람이나 지 정신 아님,0
2090,누군 자살을 종용 했을까,0
2091,HQ 중국으로 옮기면 되겠다 ㅋ 중국 게임 회사 좋네 ㅋ,0


In [ ]:
df.to_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/테스트 데이터-스마일게이트/df_test.csv')

In [ ]:
def remove_english_between_ands(text):
    return re.sub(r'& [a-zA-Z]+ &', '', text) # 이거 뒤에 결합한 문자도 지우기

df_train['input'] = df_train['input'].apply(remove_english_between_ands)
df_dev['input'] = df_dev['input'].apply(remove_english_between_ands)
df_test['input'] = df_test['input'].apply(remove_english_between_ands)

In [ ]:
def remove_urls(text):
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, '', text)

df_train['input'] = df_train['input'].apply(remove_urls)
df_dev['input'] = df_dev['input'].apply(remove_urls)
df_test['input'] = df_test['input'].apply(remove_urls)

In [ ]:
def remove_special_characters(text):
    return re.sub(r'[^\w\s]', '', text)

df_train['input'] = df_train['input'].apply(remove_special_characters)
df_dev['input'] = df_dev['input'].apply(remove_special_characters)
df_test['input'] = df_test['input'].apply(remove_special_characters)

In [ ]:
def remove_repeated_characters(text):
    return re.sub(r'(.)\1+', r'\1', text)

df_train['input'] = df_train['input'].apply(remove_repeated_characters)
df_dev['input'] = df_dev['input'].apply(remove_repeated_characters)
df_test['input'] = df_test['input'].apply(remove_repeated_characters)

In [ ]:
def remove_duplicate_rows(df, column_name):
    df.drop_duplicates(subset=column_name, inplace=True)
remove_duplicate_rows(df_train, 'input')
remove_duplicate_rows(df_dev, 'input')

In [ ]:
df_train.to_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_train_pre.csv')
df_dev.to_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_dev_pre.csv')
df_test.to_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_test_pre.csv')

In [5]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=32
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')
df_train_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_train_pre.csv', encoding = 'utf-8')
df_dev_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_dev_pre.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_test_pre.csv', encoding = 'utf-8')

df_train_pre.dropna(inplace = True)
df_dev_pre.dropna(inplace = True)
df_test_pre.dropna(inplace = True)

df_train_pre.reset_index(drop = True, inplace = True)
df_dev_pre.reset_index(drop = True, inplace = True)

cuda


In [6]:
df_train_pre.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis = 1, inplace = True)
df_dev_pre.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis = 1, inplace = True)
df_test_pre.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis = 1, inplace = True)
df_test.drop(['Unnamed: 0'], axis = 1, inplace = True)

In [7]:
from tokenizers.processors import TemplateProcessing

class CustomDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

model_name = 'EleutherAI/polyglot-ko-1.3b'
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True)
tokenizer.pad_token = tokenizer.eos_token

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch])  # 레이블을 Tensor로 변환

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks, labels

train_dataset = CustomDataset(df_train_pre, 'input', 'output', tokenizer, args.max_len)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch)
valid_dataset = CustomDataset(df_dev_pre, 'input', 'output', tokenizer, args.max_len)
valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

tokenizer_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size, output_attentions=False, output_hidden_states=False)
model.to(device)

model.config.pad_token_id = tokenizer.pad_token_id

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=args.start_lr)
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                            num_training_steps = len(train_loader)*args.epochs) # cosine annealing

def train(model, data_loader, loss_fn, optimizer, scheduler):
    train_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    model.train()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())
        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / i, 4)))
    train_loss = train_loss / len(data_loader)
    train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return model, train_loss, train_score, train_acc

def valid(model, data_loader, loss_fn):
    valid_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    model.eval()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        output = model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        valid_loss += loss.item()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())
        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / i, 4)))
    valid_loss = valid_loss / len(data_loader)
    valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return model, valid_loss, valid_score, valid_acc

stop_val_f1 = 0
stop_count = 0

for epoch in range(args.epochs):
    if stop_count == 3:
        break

    model, train_loss, train_score, train_acc = train(model, train_loader, loss_fn, optimizer, scheduler)
    model, valid_loss, valid_score, valid_acc = valid(model, valid_loader, loss_fn)

    print(f'Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
    print(f'              v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

    if valid_score > stop_val_f1:
        default_weight_path = path + 'ddd_polyglot_ko_t_e.pt'
        torch.save(model.state_dict(), default_weight_path)
        stop_val_f1 = valid_score
        stop_count = 0
    else:
        stop_count += 1

    torch.cuda.empty_cache()
    gc.collect()

config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/31.6k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/748M [00:00<?, ?B/s]

...

Epoch : 8,    t_loss : 0.0001,   t_f1 : 1.0,   t_acc : 1.0

              v_loss : 0.3875,   v_f1 : 0.9375,   v_acc : 0.9396



[C_loss : 0.401]: 100%|██████████| 65/65 [00:20<00:00,  3.22it/s]


Epoch : 9,    t_loss : 0.0001,   t_f1 : 1.0,   t_acc : 1.0

              v_loss : 0.401,   v_f1 : 0.9375,   v_acc : 0.9396



[C_loss : 0.4054]: 100%|██████████| 65/65 [00:20<00:00,  3.22it/s]


Epoch : 10,    t_loss : 0.0001,   t_f1 : 1.0,   t_acc : 1.0

              v_loss : 0.4054,   v_f1 : 0.937,   v_acc : 0.9391



In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks

tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True)
tokenizer.pad_token = tokenizer.eos_token
test_dataset = TestDataset(df_test_pre, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size, output_attentions=False, output_hidden_states=False)
model.config.pad_token_id = tokenizer.pad_token_id

model.load_state_dict(torch.load('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_polyglot_ko_t_e.pt', map_location=device))
model.to(device)

model.eval()
res = []
with torch.no_grad():
    for t, data in enumerate(tqdm(test_loader)):
        ids = data[0].to(device)
        mask = data[1].to(device)

        output = model(ids, mask)[0]
        output = output.cpu().numpy()
        res.extend(output.argmax(axis = 1).tolist())

df_test['test_output'] = res

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-1.3b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 65/65 [00:19<00:00,  3.36it/s]


In [ ]:
df_test

,id,input,output
0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...
2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/결과파일/test3_polyglot_ko.jsonl')

In [9]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=32
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'
model_name = 'EleutherAI/polyglot-ko-1.3b'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')
# df_train_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_train_pre.csv', encoding = 'utf-8')
# df_dev_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_dev_pre.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/테스트 데이터-스마일게이트/df_test.csv', encoding = 'utf-8')

# df_train_pre.dropna(inplace = True)
# df_dev_pre.dropna(inplace = True)
df_test_pre.dropna(inplace = True)

# df_train_pre.reset_index(drop = True, inplace = True)
# df_dev_pre.reset_index(drop = True, inplace = True)

cuda


In [10]:
from tokenizers.processors import TemplateProcessing

class TestDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

model_name = 'EleutherAI/polyglot-ko-1.3b'
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True)
tokenizer.pad_token = tokenizer.eos_token

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch])  # 레이블을 Tensor로 변환

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks, labels

test_dataset = TestDataset(df_test_pre, 'input', 'output', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size, output_attentions=False, output_hidden_states=False)
model.load_state_dict(torch.load('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_polyglot_ko_t_e.pt', map_location=device))
model.to(device)
model.config.pad_token_id = tokenizer.pad_token_id

def evaluate_model(model, data_loader):
    target_lst = []
    pred_lst = []
    model.eval()
    with torch.no_grad():
        for ids, mask, target in tqdm(data_loader):
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            output = model(ids, mask)
            pred_lst.extend(output.logits.argmax(dim=1).tolist())
            target_lst.extend(target.cpu().numpy())

    eval_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    eval_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    return eval_score, eval_acc

f1_score, accuracy = evaluate_model(model, test_loader)
print(f"F1-Score: {f1_score}, Accuracy: {accuracy}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-1.3b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 66/66 [00:20<00:00,  3.26it/s]

F1-Score: 0.7986803124158364, Accuracy: 0.7993311036789298
